In [ ]:
# see link
# https://github.com/Fraud-Detection-Handbook

package ='07-sklearn.svm'
name='SVC-kernel-02poly'
tuningAndParameters='03-Random Undersampling'

hyperparametersFound = {'kernel': 'poly', 'C': 1, 'degree':4}
scalerFound='StandardScaler'
rateFound=45

print('done')

In [ ]:
import sys
import os
from importlib import reload
fpath = os.path.join('..//scripts')
sys.path.append(fpath)

import warnings
warnings.filterwarnings('ignore')

#loading internal scripts
import datamanagement as dm
reload(dm)

import result as resultMd
reload(resultMd)

import graph as gf
reload(gf)

import scaler as scaler
reload(scaler)

print('done')

In [ ]:
dfLearning, dfValidation =dm.getDataLearningAndValidation()

dfLearning.head()

# Scaling choice

In [ ]:
%%script false

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()
scalingData=[]

scalers = scaler.getScalers()
for key in scalers:
    print(key)
    x1, y1 = dfLearning[predictors], dfLearning[target]
    sc=scalers.get(key)
    x2 = sc.fit_transform(x1)

    TEST_SIZE = 0.20 # test size using_train_test_split
    RANDOM_STATE = 0


    x_train0, x_test, y_train0, y_test = train_test_split(x2, y1, test_size = TEST_SIZE, 
                                                        stratify=y1,
                                                        random_state = RANDOM_STATE)


   
    modelClf = SVC(kernel="poly",random_state=42)
    parameters=hyperparametersFound
    modelClf.set_params(**parameters)

    modelClf.fit(x_train0, y_train0)
    predsTrain = modelClf.predict(x_train0)
    predsTest = modelClf.predict(x_test)

    train_f1=f1_score(y_train0, predsTrain)
    print("f1 train {:.4f}".format(train_f1))
    
    test_f1=f1_score(y_test, predsTest)
    print("f1 test  {:.4f}".format(test_f1))
    print('-----------------------')
    subScalingData = [train_f1,test_f1]
    scalingData.append(subScalingData)

print(scalingData)
import matplotlib.pyplot as plt

fig = plt.figure(figsize =(10, 7))
ax = fig.add_subplot(111)
bp = ax.boxplot(scalingData, patch_artist = True,
                notch ='True', vert = 1)
ax.set_xticklabels(['StandardScaler','MinMaxScaler','RobustScaler','MaxAbsScaler'])
plt.title("Scaling choice")
ax.get_xaxis().tick_bottom()
ax.get_yaxis().tick_left()
plt.show()

# Sampling choice

In [ ]:
%%script false

from sklearn.svm import SVC
import matplotlib.pyplot as plt
from datetime import datetime
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_val_score
import numpy as np
from sklearn.metrics import f1_score

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

train_f1s=[]
test_f1s =[]
range= []

x1, y1 = dfLearning[predictors], dfLearning[target]
sc = scaler.getScaler(scalerFound)
x2 = sc.fit_transform(x1)    

TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0

x_train0, x_test, y_train0, y_test = train_test_split(x2, dfLearning[target], test_size = TEST_SIZE, 
                                                        stratify=dfLearning[target],
                                                        random_state = RANDOM_STATE)

rates = np.arange(5,80,5)
for rate in rates:
    print("",rate,1/rate)
    undersample = RandomUnderSampler(sampling_strategy=1/rate,random_state=42)
    x_train, y_train = undersample.fit_resample(x_train0, y_train0)

    print(x_train0.shape)
    print(x_train.shape)
    #print(y_train.value_counts())
    
    modelClf = SVC(kernel="poly",random_state=42)
    parameters=hyperparametersFound
    modelClf.set_params(**parameters)

    modelClf.fit(x_train, y_train)
    predsTrain = modelClf.predict(x_train)
    predsTest = modelClf.predict(x_test)

    train_f1=f1_score(y_train, predsTrain)
    print("f1 train {:.4f}".format(train_f1))
    
    test_f1=f1_score(y_test, predsTest)
    print("f1 test  {:.4f}".format(test_f1))
    
    train_f1s.append(train_f1)
    test_f1s.append(test_f1)
    range.append(rate)
    print('-----------------------')

dm.plt_train_test(range, train_f1s, "train", test_f1s," test")

#rate found = 10

# Hyperparameters tuning

In [ ]:
from imblearn.under_sampling import NearMiss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.under_sampling import RandomUnderSampler


TEST_SIZE = 0.20 # test size using_train_test_split
RANDOM_STATE = 0

predictors = dm.getPredictors(dfLearning)
target = dm.getTarget()

## Scaling
x1, y1 = dfLearning[predictors], dfLearning[target]
sc = scaler.getScaler(scalerFound)
x2 = sc.fit_transform(x1)


x_train0, x_test, y_train0, y_test = train_test_split(x2, dfLearning[target], test_size = TEST_SIZE, 
                                                        stratify=dfLearning[target],
                                                        random_state = RANDOM_STATE)

## Sampling
rate=rateFound
undersample = RandomUnderSampler(sampling_strategy=1/rate,random_state=42)
x_train, y_train = undersample.fit_resample(x_train0, y_train0)

## Randomized Search 

In [ ]:
%%script false

from sklearn.svm import SVC
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV


dic_param={
    'C': [0.001,0.01,0.1, 1, 10, 100], 
	'degree':randint(2,5)
}
modelClf = SVC(kernel="poly",random_state=42)
random_search = RandomizedSearchCV(modelClf,dic_param, scoring='f1', verbose=10,cv=4,n_iter=5).fit(x_train, y_train)

print(random_search.best_params_)
print(random_search.best_score_)


#{'C': 1, 'degree': 4}
#0.770080571476916

#{'C': 10, 'degree': 3}
#0.779828594218788

#{'C': 100, 'degree': 2}
#0.7179524032512674


## Bayes Search

## Grid Search

In [ ]:
%%script false

from sklearn.svm import SVC
from scipy.stats import randint
from sklearn.model_selection import GridSearchCV
import numpy as np


dic_param={
    'C': [0.001,0.01,0.1, 1, 10, 100], 
	'degree':[2,3,4]
}
modelClf = SVC(kernel="poly",random_state=42)

random_search = GridSearchCV(modelClf,dic_param, scoring='f1', verbose=10,cv=4).fit(x_train, y_train)
print(random_search.best_params_)
print(random_search.best_score_)

#{'C': 10, 'degree': 3}
#0.779828594218788

#{'C': 1, 'degree': 4}
#0.7204141114630366


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.svm import SVC
from datetime import datetime

modelClf = SVC(kernel="poly",random_state=42)
parameters=hyperparametersFound

modelClf.set_params(**parameters)
then= datetime.now()
modelClf.fit(x_train, y_train)
now = datetime.now()
duration= now - then
learningDurationInS = duration.total_seconds()
print("Duration ",learningDurationInS )
resultMd.update_time_response_result(package, name, tuningAndParameters, learningDurationInS)

predsTrain = modelClf.predict(x_train)
predsTest = modelClf.predict(x_test)

f1Learning =f1_score(y_train, predsTrain)
f1Test=f1_score(y_test, predsTest)
dm.show_confusion_matrix(y_train, predsTrain,'Confusion matrix learning data')
print(f"f1 train {f1Learning:.3f}")
dm.show_confusion_matrix(y_test, predsTest,'Confusion matrix test data')
print(f"f1 test {f1Test:.3f}")
resultMd.update_learning_test_result(package, name, tuningAndParameters, f1Learning,f1Test)

In [ ]:
gf.show_importance(modelClf, predictors)

In [ ]:
# does not work with SVC
#gf.show_prediction_graph(modelClf, x_test,y_test)

In [ ]:
dfValidatationScaled = sc.transform(dfValidation[predictors])

predsValidation = modelClf.predict(dfValidatationScaled)
f1Validation=f1_score(dfValidation[target], predsValidation)

dm.show_confusion_matrix(dfValidation[target], predsValidation,'Confusion matrix validation data')
print(f"f1 validation {f1Validation:.3f}")
resultMd.update_performance_result(package, name, tuningAndParameters, f1Validation)
resultMd.update_hyperparameters_result(package, name, tuningAndParameters, hyperparametersFound,scalerFound)


# Summary

In [ ]:
print('Summary')
print(f"{package} {name} {tuningAndParameters}") 
print(f"hyperparameters {hyperparametersFound}") 
print(f"scaler {scalerFound}") 
print('-----------------------------')

print(f"learning duration {learningDurationInS:.2f} s")
print('-----------------------------')
print(f"f1 train      {f1Learning:.3f}")
print(f"f1 test       {f1Test:.3f}")
print(f"f1 validation {f1Validation:.3f}")